# Medical Diagnosis RAG — coursework notebook

*Original project notebook, as run during graduate coursework (Google Colab).*
Colab-specific setup (Drive mounting, `!pip install`) has been removed — install dependencies with
`pip install -r requirements.txt` and point `MEDICAL_PDF_PATH` at your copy of the manual (see `data/README.md`).

The productionized version of this pipeline lives in `src/` (no LangChain dependency), with a CLI
(`scripts/ask.py`), a Streamlit demo (`app/streamlit_app.py`), and unit tests.


## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

### Environment setup

This notebook was originally run on Google Colab. To run it locally:

```bash
pip install -r requirements.txt
export MEDICAL_PDF_PATH=/path/to/medical_diagnosis_manual.pdf
```


In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
#import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [2]:
## Model configuration
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
lcpp_llm = Llama(
    model_path=model_path,
    n_threads=2,  # CPU cores
    n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    n_gpu_layers=43,  # Change this value based on your model and your GPU VRAM pool.
    n_ctx=4096,  # Context window
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [5]:
# function to generate, process, and return the response from the LLM
def generate_llama_response(user_prompt):

    # System message
    system_message = """
    [INST]<<SYS>> Respond to the user question based on the user prompt<</SYS>>[/INST]
    """

    # Combine user_prompt and system_message to create the prompt
    prompt = f"{user_prompt}\n{system_message}"

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=500,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]
    return response_text

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
user_prompt = "What is the protocol for managing sepsis in a critical care unit?"
response = generate_llama_response(user_prompt)
print(response)

 The management of sepsis in a critical care unit (CCU) follows a standardized protocol that includes early recognition, aggressive fluid resuscitation, antibiotic therapy, and close monitoring. Here are the key components of the protocol:

1. Early Recognition: The CCU team should be vigilant in identifying patients at risk for sepsis, using clinical criteria such as fever, tachycardia, tachypnea, or signs of organ dysfunction. Blood cultures should be obtained promptly if sepsis is suspected.
2. Aggressive Fluid Resuscitation: Septic patients may have hypovolemia due to vasodilation and increased capillary permeability. Crystalloid fluids (e.g., lactated Ringer's solution) should be administered intravenously to restore blood pressure and perfusion.
3. Antibiotic Therapy: Broad-spectrum antibiotics effective against common causes of sepsis, such as Pseudomonas aeruginosa and Gram-negative bacteria, should be initiated promptly. The choice of antibiotics may vary depending on the susp

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [7]:
user_prompt = " What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Sure, I'd be happy to help! Appendicitis is a medical emergency that requires immediate attention. The common symptoms of appendicitis include:

1. Severe pain in the abdomen, usually starting near the belly button and then moving to the lower right side of the abdomen.
2. Nausea and vomiting.
3. Loss of appetite.
4. Fever.
5. Abdominal tenderness and guarding (muscle tension).
6. Abdominal swelling.
7. Diarrhea or constipation.

If you suspect that you or someone else may have appendicitis, it is essential to seek medical attention immediately. Appendicitis cannot be cured via medicine; instead, surgery is required to remove the inflamed appendix. The surgical procedure most commonly used to treat appendicitis is an appendectomy, which involves removing the inflamed appendix through a small incision in the abdomen.

There are two types of appendectomies: open and laparoscopic. An open appendectomy involves making a larger incision in the abdomen to access the appendix directly. A lap

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [8]:
user_prompt = " What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Sudden patchy hair loss, characterized by localized bald spots on the scalp, can be a distressing and challenging condition to address. The possible causes of this phenomenon are diverse, ranging from hormonal imbalances, autoimmune disorders, infections, allergies, and nutritional deficiencies. Here are some effective treatments or solutions that may help address sudden patchy hair loss:

1. Medications: Minoxidil (Rogaine) and finasteride (Propecia) are two FDA-approved medications that have been shown to promote hair growth and slow down hair loss. These medications work by stimulating blood flow, reducing inflammation, and blocking the conversion of testosterone to dihydrotestosterone (DHT), a hormone linked to balding.
2. Low-level laser therapy (LLLT): LLLT uses low-level lasers or light-emitting diodes to stimulate hair growth by increasing blood flow and promoting cellular activity. Studies have shown that LLLT can improve hair density, increase hair thickness, and reduce the 

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [9]:
user_prompt = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Treatments for a physical injury to brain tissue depend on the severity and location of the injury, as well as the specific symptoms and needs of the individual. Here are some common treatments that may be recommended:

1. Medications: Various medications may be prescribed to manage symptoms such as pain, inflammation, seizures, or mood changes. These can include analgesics, anti-inflammatory drugs, anticonvulsants, and mood stabilizers.
2. Rehabilitation therapy: Physical, occupational, and speech therapies may be necessary to help the individual regain lost function and improve their quality of life. These therapies can include exercises to strengthen affected muscles, activities to improve cognitive and motor skills, and training in the use of assistive devices such as walkers or wheelchairs.
3. Surgery: In some cases, surgery may be necessary to relieve pressure on brain tissue, repair damaged blood vessels, or remove a tumor or other foreign object that is causing injury.
4. Stem

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [10]:
user_prompt = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Sure, I'd be happy to help! If someone has fractured their leg during a hiking trip, it is essential to take immediate action to provide proper care and prevent further complications. Here are some necessary precautions and treatment steps that should be considered:

1. Stop activity: The first step is to stop any physical activity and rest the affected limb immediately. This will help prevent further damage or displacement of the fracture.
2. Assess the injury: Evaluate the severity of the fracture by checking for signs such as swelling, deformity, or difficulty moving the leg. If the fracture is severe or if there are any open wounds, call for emergency medical assistance immediately.
3. Immobilize the limb: Use a splint or brace to immobilize the affected leg and prevent movement. This will help reduce pain and promote healing.
4. Apply ice: Apply ice packs to the affected area to reduce swelling and relieve pain. Wrap the ice in a towel or cloth to avoid direct contact with the sk

- The answers are more general and basics.

## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [11]:
# Removed the line 'Hana' as it was causing a NameError.
user_prompt = " Provide a step-by-step clinical protocol for managing sepsis in a critical care (ICU) setting, including early identification, diagnostic criteria, antibiotic therapy, fluid resuscitation, and hemodynamic monitoring."
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Sure, here is a step-by-step clinical protocol for managing sepsis in an intensive care unit (ICU) setting:

Step 1: Early Identification and Suspicion of Sepsis
-----------------------------------------------

* Assess the patient's vital signs, including temperature, blood pressure, tachycardia, and tachypnea.
* Check for signs of infection, such as fever, purulent drainage, or wound redness.
* Evaluate the patient's medical history for risk factors for sepsis, such as recent surgery, immunosuppression, or chronic illness.

Step 2: Diagnostic Criteria for Sepsis
--------------------------------------

* Use the Systemic Inflammatory Response Syndrome (SIRS) criteria to diagnose sepsis:
	+ Body temperature >38°C or <36°C
	+ Heart rate >90 beats per minute
	+ Respiratory rate >20 breaths per minute or arterial CO2 tension (PaCO2) <32 mmHg
	+ White blood cell count (WBC) >12,000 cells/mm3 or <4,000 cells/mm3, or >10% bands (immature white blood cells)
* Confirm the presence of an infec

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [12]:
user_prompt = " What are the typical clinical symptoms of appendicitis, and under what circumstances can it be treated non-surgically with antibiotics? If surgery is required, describe the standard operative procedure and post-operative care. "
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Sure, I'd be happy to help! Appendicitis is a medical emergency that requires prompt treatment to prevent complications and potentially life-threatening outcomes. Here are the typical clinical symptoms of appendicitis, as well as information on non-surgical treatment with antibiotics and the standard operative procedure and post-operative care when surgery is required:

Typical Clinical Symptoms of Appendicitis:
------------------------------------------

The typical clinical symptoms of appendicitis include:

1. Severe pain in the lower right abdomen that begins suddenly and worsens over time. The pain may be described as a sharp, stabbing sensation or a dull ache.
2. Nausea and vomiting are common, especially in the early stages of appendicitis.
3. Loss of appetite and fever are also typical symptoms.
4. Abdominal tenderness and guarding (muscle tension) in the lower right abdomen.
5. Palpable rigidity in the abdomen, which is a sign of peritonitis (inflammation of the lining of the

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [13]:
user_prompt = " Explain the most effective treatments for sudden-onset patchy hair loss (e.g., alopecia areata), including both medical and dermatological options. What are the likely causes and diagnostic steps to confirm the condition? "
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Sure, I'd be happy to help! Sudden-onset patchy hair loss, such as alopecia areata, can be a distressing condition that affects both men and women. While there is no cure for alopecia areata, there are several effective treatments available that can help promote hair growth and improve the appearance of your hair.

Medical Treatment Options:

1. Corticosteroid Injections: These injections are directly administered into the affected area to reduce inflammation and suppress the immune system, which is believed to play a role in alopecia areata. They can help promote hair growth and slow down further hair loss.
2. Topical Corticosteroids: For mild to moderate cases of alopecia areata, topical corticosteroid creams or ointments may be prescribed to reduce inflammation and promote hair growth. These can include hydrocortisone cream, triamcinolone acetonide cream, or clobetasol propionate cream.
3. Minoxidil: This medication is applied directly to the affected area and can help stimulate ha

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [14]:
user_prompt = " What are the recommended acute and long-term treatments for a patient with traumatic brain injury (TBI), and how do these vary based on the severity of impairment (mild, moderate, or severe)? Include rehabilitation and monitoring considerations. "
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Traumatic brain injury (TBI) can have a significant impact on a patient's quality of life, cognitive functioning, and physical abilities. The recommended acute and long-term treatments for TBI vary based on the severity of impairment, which is typically classified as mild, moderate, or severe. Here are some general considerations for each severity level:

Mild TBI (concussions):

* Acute treatment: Observation and monitoring for 24-48 hours to assess for any worsening of symptoms.
* Long-term treatment: Cognitive therapy, physical therapy, and behavioral therapy may be helpful in addressing symptoms such as headaches, dizziness, and difficulty with concentration and memory.

Moderate TBI:

* Acute treatment: Hospitalization for close monitoring and management of respiratory and cardiovascular functions, as well as surgical intervention if necessary (e.g., craniotomy or brain decompression).
* Long-term treatment: Rehabilitation programs that include physical therapy, occupational ther

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [15]:
user_prompt = " Outline the first aid, diagnostic steps, and treatment plan for a leg fracture sustained during a hiking trip. What precautions should be taken in remote environments, and what does the recovery and rehabilitation process involve? "
response = generate_llama_response(user_prompt)
print(response)

Llama.generate: prefix-match hit


 Sure! Here's my response:  

If you sustain a leg fracture while hiking in a remote environment, it is crucial to seek medical attention as soon as possible. However, if you are unable to access immediate medical care, here are the first aid, diagnostic steps, and treatment plan that can help manage your injury:  

First Aid Steps: 

1. Stop any bleeding by applying direct pressure to the wound using a clean cloth or bandage. Elevate the affected limb above heart level to reduce swelling. 
2. Immobilize the fractured leg with a splint made of sturdy material such as a hiking stick, tree branch, or even a plastic water bottle. This will help maintain the alignment of the bones and prevent further damage.  
3. Apply ice packs wrapped in cloth to reduce pain and swelling. 
4. Monitor vital signs, including pulse rate, breathing rate, and blood pressure. Keep the patient warm with extra layers of clothing or a space blanket. 
5. If possible, take photographs of the injury from different a

- The answers are more specific and detailed.

## Data Preparation for RAG

### Loading the Data

In [ ]:
import os
from pathlib import Path

# Point this at your copy of the manual (see data/README.md).
PDF_PATH = Path(os.environ.get("MEDICAL_PDF_PATH", "data/medical_diagnosis_manual.pdf"))
print(PDF_PATH)


In [ ]:
pdf_loader = PyMuPDFLoader(str(PDF_PATH))


In [6]:
medical = pdf_loader.load()


### Data Overview

#### Checking the first 3 pages

In [7]:
for i in range(3):
    print(f"Page Number : {i+1}",end="\n")
    print(medical[i].page_content,end="\n")

Page Number : 1
hanamekonnen2003@gmail.com
XWQF01C793
for personal use by hanamekonnen2003
shing the contents in part or full is liable 

Page Number : 2
hanamekonnen2003@gmail.com
XWQF01C793
This file is meant for personal use by hanamekonnen2003@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .............................................................................................................................................................................

- The first 3 pages are index

#### Checking the number of pages

In [8]:
len(medical)


4114

### Data Chunking

In [9]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

In [10]:
document_chunks = pdf_loader.load_and_split(text_splitter)


In [11]:
len(document_chunks)


8492

In [12]:
document_chunks[0].page_content


'hanamekonnen2003@gmail.com\nXWQF01C793\nfor personal use by hanamekonnen2003\nshing the contents in part or full is liable'

In [13]:
document_chunks[-2].page_content


'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nhanamekonnen2003@gmail.com\nXWQF01C793\nThis file is meant for personal use by hanamekonnen2003@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [14]:
document_chunks[-1].page_content


"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

### Embedding

In [15]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [16]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [17]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

- The embedding model generates a uniform-length vector representation, regardless of how many chunks are processed.
- This consistency is useful for performing meaningful similarity comparisons between them.

### Vector Database

In [18]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [19]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [20]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [21]:
vectorstore.embeddings


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False)

In [22]:
vectorstore.similarity_search(" effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and the possible causes behind it ",k=2)


[Document(page_content='corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia areata is

- From the retrieved chunks, we observe that all the chunks are related to the key terms

### Retriever

In [23]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [24]:
rel_docs = retriever.get_relevant_documents("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?")
rel_docs

[Document(page_content='corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia areata is

- We can observe that the two relevant chunks contain the answer to the query.  
- If we increase the **`k`** value, there is a chance that we might find the answer in even more chunks.

### Response Function

In [25]:
def generate_rag_response(user_input,k=3,max_tokens=500,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Downloading and Loading the model

In [26]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [27]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [28]:
#uncomment the below snippet of code if the runtime is connected to GPU.
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [29]:
llm("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?")['choices'][0]['text']


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles, leading to hair loss in small patches on the scalp or other parts of the body. The exact cause of alopecia areata is not fully understood, but it is believed to be related to a dysfunction of the immune system that causes the body's immune cells to attack the hair follicles.\n\nThere is no single effective treatment for alopecia areata that works for everyone, as the response to treatments can vary greatly from"

- The response feels broad and possibly sourced from external content. Let's tailor it to our specific context and ensure it directly addresses our query.

## System and User Prompt Template

In [30]:
qna_system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [31]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

## Response Function

In [32]:
def generate_rag_response(user_input,k=3,max_tokens=500,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [33]:
user_input = " What is the protocol for managing sepsis in a critical care unit? "
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


I don't know. The context does not provide information on the protocol for managing sepsis in a critical care unit.


- The system prompt clearly states that the query should only be answered if the information can be found within the provided context.
- As intended, the model performed effectively by avoiding fabricated responses and staying grounded in the given data.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [34]:
user_input = " What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it? "
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


The common symptoms for appendicitis include pain in the lower right abdomen, loss of appetite, nausea or vomiting, fever, and a feeling of being sick. Appendicitis cannot be cured via medicine alone as it requires surgical removal of the appendix. The treatment for appendicitis is open or laparoscopic appendectomy.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [35]:
user_input = " What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it? "
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is Alopecia Areata. The context mentions that it is a type of hair loss characterized by sudden patchy hair loss in people with no obvious skin or systemic disorder. It is thought to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.

The context also suggests that corticosteroids, retinoids, or immunosuppressants may be used as treatments for hair loss due to various causes, including Alopecia Areata. However, it is important to note that the context does not provide specific information on the effectiveness of these treatments for Alopecia Areata or any potential side effects.

Therefore, based on the context provided, the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, could be corticosteroids, retinoids, or immunosuppressants. The possible causes behind it are thought to be an autoimmune

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [36]:
user_input = " What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function? "
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Answer: The initial treatment for a person with traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [37]:
user_input = " What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery? "
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context provided, here's the answer:

The necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip include elevating the injured limb above the heart for the first two days to minimize swelling. After 48 hours, periodic application of warmth using a heating pad for 15 to 20 minutes may relieve pain and speed healing. Immobilization is necessary to prevent further injury and facilitate healing, with joints proximal and distal to the injury also being immobilized. A cast is usually used for fractures or other injuries that require weeks of immobilization. Patients with casts should be given written instructions to keep the cast dry, never put an object inside the cast, inspect the cast's edges and skin around the cast every day and apply lotion to any red or sore areas, and pad any rough edges with soft adhesive tape, cloth, or other soft material to prevent the cast's edges. It is important to note that severe swelling under a 

- All the answers are clear, concise, and focused, without any unnecessary information.  

## Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [38]:
user_input = " What is the protocol for managing sepsis in a critical care unit? "
print(generate_rag_response(user_input, temperature=0, top_p=0, top_k=5, max_tokens=50))

Llama.generate: prefix-match hit


I don't know. The context does not provide information on the protocol for managing sepsis in a critical care unit.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [39]:
user_input = " What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it? "
print(generate_rag_response(user_input, temperature=2, top_p=1, top_k=70, max_tokens=500))

Llama.generate: prefix-match hit


From the context, an uninflamed or non-perforated appendix doesn't require any medication inappendicilis cases. Instead of medication, surgery- specifically open or laparoscopic appendectomy- is required to either completely remove an infected or ruptured appendix incase if it's diagnosed with appendicitis. Said procedure aims to delay treatment reduces mortality to a considerable extent. However, for appendicular symptoms and proper diagnosis are inevitable, medical aid provides relief yet, its entirely curative abilities lacking unlike its surgical intervention in counterpart.The possible complicating scenarios of late presentation and formation involving mass affecting pericolelce (appendix site of inflammation, a normally harmless tubinglike appendx in the colon of lower right abdominal wall) include symptoms such re operation at later phases but can escalate to abscesses requiring specialized surgical management and sometimes an extended recovery period (especially,  long - term b

- Even if the max_tokens is set to 500, the model responces more longer detailed answers which is a little slower and costlier.
- Temperature is set to 2, making the model more generate incoherent or offf-topic text.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [40]:
user_input = " What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it? "
print(generate_rag_response(user_input, temperature=0.5, top_p=0.9, top_k=60, max_tokens=500))

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is Alopecia Areata. The context mentions that this is a type of hair loss characterized by sudden, patchy hair loss in people with no obvious skin or systemic disorder. It also states that Alopecia Areata is thought to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.

The context does not provide information on specific treatments or solutions for Alopecia Areata, but it does mention that corticosteroids, retinoids, or immunosuppressants may be used for treating hair loss in general. Therefore, based on the context, I cannot provide a definitive answer to what the effective treatments or solutions are for addressing sudden patchy hair loss specifically caused by Alopecia Areata. However, I can suggest that these types of medications might be used as potential treatment options.

As for possible causes behind sudden patchy hair loss, the context states that it is an 

- Even if the max_tokens is set to 500, the model still didn't generate that many, as the query could be answered with a limited availiablity of inside the context.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [41]:
user_input = " What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function? "
print(generate_rag_response(user_input, temperature=0.3, top_p=0.2, top_k=15, max_tokens=150))

Llama.generate: prefix-match hit


Answer: The initial treatment for a person with traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.


- The temperature and others are set to less, making the model more deterministic and less creative.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [42]:
user_input = " What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery? "
print(generate_rag_response(user_input, temperature=1.5, top_p=0.7, top_k=40, max_tokens=250))

Llama.generate: prefix-match hit


To answer this question, consider the following points from the context:
1. The injured limb should be elevated above the heart for the first two days to minimize swelling and help drain edema fluid.
2. After 48 hours, periodic application of warmth using a heating pad may relieve pain and speed up healing.
3. Immobilization is important to prevent further injury and facilitate healing, but care should be taken to ensure joints proximal and distal to the injury are also immobilized.
4. A cast is usually used for fractures or injuries that require weeks of immobilization. In severe cases where swelling may contribute to compartment syndrome, the cast (and all padding) may be cut open from end to end medially and laterally.
5. Patients with casts should be given written instructions on how to care for their cast, including keeping it dry, never putting objects inside it, inspecting the edges and skin around the cast daily, and padding any rough edges to prevent irritation.


## Output Evaluation

In [43]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [44]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [45]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [46]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

## Fine-tuning Evaluating

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [47]:
user_input = " What is the protocol for managing sepsis in a critical care unit? "
#print(generate_rag_response(user_input, temperature=0, top_p=0, top_k=5, max_tokens=50))

ground,rel = generate_ground_relevance_response(user_input, temperature=0, top_p=0, top_k=5, max_tokens=50)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify if the answer is derived only from the information presented in the context.
2. Check if the context contains any information related to the protocol for managing sepsis in a critical care unit

 Steps to evaluate the context as per the relevance metric:
1. Identify the main aspects of the question, which are "managing sepsis in a critical care unit."
2. Search through the context for any information related


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [48]:
user_input = " What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it? "
#print(generate_rag_response(user_input, temperature=2, top_p=1, top_k=70, max_tokens=500))

ground,rel = generate_ground_relevance_response(user_input, temperature=2, top_p=1, top_k=70, max_tokens=250)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Step 1: Identify the main components of the question and context.question components: 1. "What are the common symptoms for appendicitis?” - A descriptive question asks for information about Appendicitis symptoms2. "Can it be cured via medicine?” - Inquiry into curability via medical treatment3. "If not, what surgical procedure should be followed to treat it?” -query with regard to alternative therapeutic procedures context components:1- Information on treatment options & processes for appendicitis (open or laparoscopic appendectomy), requirements and decision making considerations surrounding antbitiotics prescriptions2. Different approaches to performing an appendectomy if the condition worsens ("In late cases with periocolic abscess etc")3- Other health conditions such as inflamtory bowel disease contrindicating a potentially definitive surgical process (appendectomy)but may indicate treatments varying based-on given medical context

Analyzing:
Despite some commonalities in vocabula

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [49]:
user_input = " What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it? "
#print(generate_rag_response(user_input, temperature=0.5, top_p=0.9, top_k=60, max_tokens=500))

ground,rel = generate_ground_relevance_response(user_input, temperature=0.5, top_p=0.9, top_k=60, max_tokens=500)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to effective treatments or solutions for sudden patchy hair loss.
2. Determine if the AI generated answer only includes information derived from the context.
3. Check if any additional information not present in the context is included in the answer.

Explanation:
The AI generated answer mentions that corticosteroids, retinoids, or immunosuppressants are effective treatments for sudden patchy hair loss, which is consistent with the information provided in the context. Additionally, it explains that these treatments help reduce inflammation and suppress the immune system's attack on the hair follicles, further elaborating on how they work to address the issue. The answer also mentions that microscopic hair examination or scalp biopsy may be required for definitive diagnosis, which is also present in the context.

Therefore, based on the given information, it can be concluded that the AI generated answer ad

- It got a perfect score because the response is both grounded in the context and relevant to the query.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [50]:
user_input = " What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function? "
#print(generate_rag_response(user_input, temperature=0.3, top_p=0.2, top_k=15, max_tokens=150))

ground,rel = generate_ground_relevance_response(user_input, temperature=0.3, top_p=0.2, top_k=15, max_tokens=150)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information in the context related to treatments for a person with a traumatic brain injury (TBI).
2. Compare the identified information with the AI generated answer to check if the answer is derived only from the context.
3. Evaluate the extent to which the metric is followed.

Explanation:
The context provides information about the initial treatment for a person with a TBI, including ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and the possibility of surgery in more severe cases. The AI generated answer matches this information exactly, stating that the initial treatment includes these steps and that surgery may be needed in

 Steps to evaluate the context as per the relevance metric:
1. Identify the main aspects of the question: The question asks about treatments for a person with a brain injury that results in temporary or permanent impairment of brain function.
2. Determine if the 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [51]:
user_input = " What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery? "
#print(generate_rag_response(user_input, temperature=1.5, top_p=0.7, top_k=40, max_tokens=250))

#ground,rel = generate_ground_relevance_response(user_input, temperature=1.5, top_p=0.7, top_k=40, max_tokens=250)
ground,rel = generate_ground_relevance_response(user_input, max_tokens=250)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context that relates to the necessary precautions and treatment steps for a person with a fractured leg.
2. Compare the information identified in step 1 to the AI generated answer to ensure that all elements of the answer are derived from the context.
3. Evaluate the extent to which the metric is followed by checking if there is any additional or extraneous information in the answer that is not present in the context.

Explanation:
The AI generated answer includes all the necessary precautions and treatment steps for a person with a fractured leg as presented in the context. The answer accurately states that the injured limb should be elevated above heart level for the first two days, application of warmth with a heating pad after 48 hours, immobilization of joints proximal and distal to the injury, and instructions for patients with casts such as keeping it dry, never putting objects inside it, inspecting edges and skin

## Actionable Insights and Business Recommendations

- The time required to build a vector database scales with the number of pages in a PDF file—larger documents take longer to process.

- The retrieval setting k plays a vital role in capturing comprehensive answers, especially when relevant content is distributed across several sections.

- To maintain contextual flow between text segments, using chunk_overlap is essential.

- The max_tokens parameter should be aligned with query complexity—complex queries benefit from a higher limit, whereas simple queries tend to yield brief responses regardless of the token ceiling, particularly when using a zero temperature setting.

- Effective management of prompt structure and temperature values helps fine-tune response length and originality.

- Continuously calibrate Retrieval-Augmented Generation (RAG) parameters to match the unique requirements of different scenarios for best results.

- Emphasize accuracy and contextual relevance when evaluating outputs to ensure dependable responses.

- Implement a feedback mechanism to iteratively optimize parameters, enhancing system performance across various query formats.

<font size=6 color='blue'>Power Ahead</font>
___